In [0]:
adls_path = "abfss://datalake@adlsfeb26batch.dfs.core.windows.net/bronze/salesteam/india/csv/ServiceNow_extracts.csv"

df = (spark.read
      .format("csv")
      .option("header", True)
      .option("inferSchema", True)
      .load(adls_path)
     )

display(df)

In [0]:
df.createOrReplaceTempView("servicenow_event_log_temp")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW servicenow_event_log_temp_deduped AS
SELECT *
FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY number, incident_state, sys_updated_at
           ORDER BY sys_updated_at DESC
         ) AS rn
  FROM servicenow_event_log_temp
)
WHERE rn = 1

In [0]:
%sql
SELECT *
FROM servicenow_event_log_temp

In [0]:
%sql
MERGE INTO project1catalog.edw_silver.servicenow_event_log AS target
USING servicenow_event_log_temp_deduped AS source
ON target.number = source.number
AND target.incident_state = source.incident_state
AND target.sys_updated_at = source.sys_updated_at
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *